In [ ]:
import numpy as np
import pandas as pd

**Preparing Dataset**

In [ ]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

movies = movies.merge(credits,on='title')

**Data Preprocessing**

In [ ]:
import ast

movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]
movies.isnull().sum()
movies.dropna(inplace=True)
movies.duplicated().sum()

def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L
#covert format of 'genres' and 'keywords'
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

def convert3(obj):
    L = []
    for i in ast.literal_eval(obj):
      if len(L) < 3:
        L.append(i['name'])
      else:
        break
    return L
#selecting first 3 casts
movies['cast'] = movies['cast'].apply(convert3)

def fetch_director(obj):
    L = []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
    return L
#extracting director name
movies['crew'] = movies['crew'].apply(fetch_director)

#converting 'overview' from string to list
movies['overview'] = movies['overview'].apply(lambda x:x.split())

#removing spaces between words
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])

#preparing new column 'tags'
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

#final dataset containing only title, id, and tags
new_df = movies[['movie_id','title','tags']]
#adding spaces between words in 'tags'
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))
#converting 'tags' in lowercase
new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())
new_df.head()

#converting all (tenses) words into root word
import nltk
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

new_df['tags'] = new_df['tags'].apply(stem)

/tmp/ipython-input-1145169432.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))
/tmp/ipython-input-1145169432.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())
/tmp/ipython-input-1145169432.py:69: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http

**Model Building**

In [ ]:
import sklearn
from sklearn.feature_extraction.text import CountVectorizer
#counting max features
cv = CountVectorizer(max_features=5000,stop_words='english')
#converting into numpy array
vectors = cv.fit_transform(new_df['tags']).toarray()

#distance between vectors (cosine)
import sklearn
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vectors)
#sorted distances in desc order and selecting top 5 index
sorted(list(enumerate(similarity[0])),reverse=True,key=lambda x:x[1])[1:6]


#recommending movies
def recommend(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:6]
    #print movies name
    for i in movies_list:
        print(new_df.iloc[i[0]].title)

recommend('Avatar')

Aliens vs Predator: Requiem
Aliens
Falcon Rising
Independence Day
Titan A.E.


In [ ]:
import pickle
pickle.dump(new_df,open('movies.pkl','wb'))
pickle.dump(similarity,open('similarity.pkl','wb'))
pickle.dump(new_df.to_dict(),open('movie_dict.pkl','wb'))